# rag query

In [2]:
import os
os.chdir(r"C:\Users\OWNER\Desktop\STAJ MEMO\RAG\memorial_rag+sql")   # Bölüm 1 ile AYNI klasör
print("Çalışma dizini:", os.getcwd())

# --- Veritabanına bağlanmak için (Bölüm 1 ile aynı olmak ZORUNDA) ---
CHROMA_DIR = "vector_db"
COLLECTION_NAME = "belgelerim"
EMBEDDING_MODEL = "BAAI/bge-m3"

# --- Sorgu davranışı ---
TOP_K = 5                                   # kaç chunk çekilecek

# --- Cevap üreten LLM ---
OLLAMA_MODEL = "qwen3:8b"   # modeli değiştirdim
OLLAMA_HOST = "http://localhost:11434"

RAG_PROMPT_TEMPLATE = """Aşağıda verilen BAĞLAM bilgisini kullanarak kullanıcının SORUSUNU yanıtla.

Kurallar:
- Cevabını öncelikle BAĞLAM'daki bilgilere dayandır.
- BAĞLAM'da doğrudan yazmasa bile, verilen bilgilerden mantıklı ve açık bir çıkarım yapılabiliyorsa bu çıkarımı yap ve hangi bilgiye dayandığını belirt.
- Sadece BAĞLAM'da hiçbir ilgili bilgi yoksa "Bu bilgi elimdeki belgelerde yok" de.
- BAĞLAM'da yer almayan sayı, isim, tarih gibi somut bilgileri kesinlikle uydurma.

BAĞLAM:
{context}

SORU:
{question}

CEVAP:"""

print("Sorgu ayarları yüklendi.")

Çalışma dizini: C:\Users\OWNER\Desktop\STAJ MEMO\RAG\memorial_rag+sql
Sorgu ayarları yüklendi.


In [3]:
from sentence_transformers import SentenceTransformer

print(f"Embedding modeli yükleniyor: {EMBEDDING_MODEL} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print("Model hazır.")

Embedding modeli yükleniyor: BAAI/bge-m3 ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model hazır.


In [4]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

# create_collection DEĞİL -> get_collection: var olan koleksiyonu açar, silmez.
collection = client.get_collection(name=COLLECTION_NAME)

print(f"Bağlanıldı: '{COLLECTION_NAME}' koleksiyonu, {collection.count()} chunk içeriyor.")

Bağlanıldı: 'belgelerim' koleksiyonu, 94 chunk içeriyor.


In [5]:
import requests

def retrieve(query, top_k=TOP_K):   # en yakın chunkları bul
    query_embedding = embed_model.encode([query], normalize_embeddings=True)[0].tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    return list(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]))

def kaynak_etiketi(meta):
    """Chunk'ın nereden geldiğini okunabilir tek satıra çevirir.
    Excel chunk'larında sayfa/satır da eklenir (bu bilgi metnin içinde değil,
    embedding'i kirletmesin diye sadece metadata'da duruyor)."""
    etiket = meta["source"]
    if "sheet" in meta:
        etiket += f" | Sayfa: {meta['sheet']} | Satır: {meta['row']}"
    return etiket

def build_context(retrieved):# bağlamı contexti olusturuyo
    parts = []
    for i, (doc, meta, dist) in enumerate(retrieved, start=1):
        parts.append(f"[Parça {i} - Kaynak: {kaynak_etiketi(meta)}]\n{doc}")
    return "\n\n".join(parts)

def call_ollama(prompt, temperature=0.3):
    response = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "think": False,
            "options": {"temperature": temperature}
        },
        timeout=600,
    )
    response.raise_for_status()
    return response.json()["response"]

def answer_question(query):
    print(f"[1/3] Sorgu embedding'e çevriliyor ve Chroma'da en yakın {TOP_K} parça aranıyor...")
    retrieved = retrieve(query)
    print(f"      -> {len(retrieved)} parça bulundu.")

    print(f"[2/3] Bağlam ve prompt oluşturuluyor...")
    context = build_context(retrieved)
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=query)

    print(f"[3/3] Ollama'ya gönderiliyor (model: {OLLAMA_MODEL})... Bu adım uzun sürebilir.")
    answer = call_ollama(prompt)
    print(f"      -> Cevap alındı.\n")

    flag = None
    if "bu bilgi elimdeki belgelerde yok" in answer.lower():
        flag = "BİLGİ_BULUNAMADI"

    return answer, flag

print("RAG fonksiyonları hazır.")

RAG fonksiyonları hazır.


In [6]:
answer, flag = answer_question("Uzm. Dr. Aslı Gökçe0 hakkında detaylı bilgi ver")
print("CEVAP:\n", answer)
if flag:
    print(f"[FLAG: {flag}]")

[1/3] Sorgu embedding'e çevriliyor ve Chroma'da en yakın 5 parça aranıyor...
      -> 5 parça bulundu.
[2/3] Bağlam ve prompt oluşturuluyor...
[3/3] Ollama'ya gönderiliyor (model: qwen3:8b)... Bu adım uzun sürebilir.
      -> Cevap alındı.

CEVAP:
 Uzm. Dr. Aslı Gökçe, Nöroloji uzmanlığı alanlarında görev yapmaktadır. Görev süresi 9 yıldır ve görevine 22.08.2016 tarihinde başlamıştır. Şubesi Marmara Sağlık Merkezi'ndedir ve şube kodu SB02'dir. Yıllık izin hakkına 20 gün, kullanılmış izin 11 gün, kalan izin ise 9 gündür. Haftalık nöbeti 5'tür ve poliklinik günleri Çarşamba - Cuma'dır. Hasta memnuniyeti oranı %88'dir. E-posta adresi gokce.d004@hastane.com.tr'dir. 2026 yılında toplam 13 işlem yapmıştır. Yapılan işlemler arasında EEG izlemi, inme trombolizi, migren botoks uygulaması, nörolojik rehabilitasyon planı, uyku testi, lomber ponksiyon ve karotis doppler yer almaktadır.


# sql query 

In [7]:
import re
import sqlite3
from contextlib import closing
from smolagents import tool, CodeAgent, LiteLLMModel

DB_URI = "file:hastane_v2.db?mode=ro"   # read-only: yazma baglanti duzeyinde reddedilir

GIZLI_SUTUNLAR = {"kaynak_dosya", "yuklenme", "hizmetler_json"}

# ─────────────────────────────────────────────────────────────────────────────
#  1) SUTUN ANLAMLARI
# ─────────────────────────────────────────────────────────────────────────────
SUTUN_ACIKLAMA = {
    "gorev_yil":       "KAC YILDIR gorevde = TECRUBE / DENEYIM / KIDEM olcusu",
    "goreve_baslama":  "ise baslama tarihi; gorev_yil bundan turetilmis",
    "unvan":           "akademik unvan",
    "toplam_islem":    "yapilan islem ADEDI = IS HACMI. Tecrube DEGIL",
    "islem_sayisi":    "yapilan islem ADEDI = IS HACMI / verimlilik. Tecrube DEGIL",
    "yatis_sayisi":    "takip edilen yatis ADEDI = IS HACMI",
    "toplam_ciro":     "doktorun yatislarinin fatura toplami (TL)",

    # --- IZIN vs NOBET ---
    "izin_hakki":         "yillik IZIN hakki (gun)",
    "izin_kullanilan":    "kullanilan IZIN (gun)",
    "izin_kalan":         "kalan IZIN (gun) = izin_hakki - izin_kullanilan",
    "haftalik_nobet":     "haftalik NOBET sayisi. IZIN DEGIL, 52 ile carpma",
    "poliklinik_gunleri": "serbest metin, or. 'Pzt-Car'",

    # --- ad / brans ---
    "ad_soyad":             "unvan dahil tam ad -> isimle ararken LIKE '%soyad%'",
    "uzmanlik":             "DOKTORUN bransi (degerleri asagida)",
    "servis":               "YATISIN servisi; uzmanlik metniyle BIREBIR AYNI DEGIL",
    "doktorlar.memnuniyet": "hasta memnuniyeti % (83-96) = 'basarili/begenilen'",
    "subeler.memnuniyet":   "subenin memnuniyet puani (%)",

    # --- SUBE: GERCEK olcek. Ornek tablolarla ayni birimde DEGIL ---
    "subeler.doktor_sayisi":   "GERCEK kadro (36-128). doktorlar tablosu 40 satirlik ORNEK",
    "subeler.toplam_personel": "GERCEK personel; ornek tablolarla kiyaslama",
    "subeler.yillik_hasta":    "GERCEK yillik hasta; yatislar tablosuyla kiyaslama",
    "subeler.yillik_ciro":     "GERCEK yillik ciro (milyar TL); SUM(fatura) ile AYNI OLCEK DEGIL",
    "subeler.net_kar":         "GERCEK yillik net kar (TL)",

    # --- yatis ---
    "yatis_id":     "bir ZIYARETIN kimligi (kisi degil)",
    "giris_tarihi": "ISO 'YYYY-MM-DD'; ay filtresi LIKE '2026-03%'",
    "cikis_tarihi": "ISO 'YYYY-MM-DD'",
    "fatura":       "TEK yatisin fatura tutari (TL)",
    "yas":          "hasta yasi; 0 = yenidogan (ortalamada dikkat)",

    # --- olay tablolari: isimleri benzer, baglantilari YOK ---
    "hizmetler.hizmet":       "BIR YATISTA verilen hizmet (yatis_id'ye bagli)",
    "doktor_islemleri.islem": "BIR DOKTORUN yaptigi islem; hizmetler ile BAGLANTISIZ",
}

# ─────────────────────────────────────────────────────────────────────────────
#  2) DEGER LISTESI (value grounding) — semanin sonuna TEK blok halinde eklenir.
# ─────────────────────────────────────────────────────────────────────────────
DEGER_LISTE_SINIRI = 20          # bundan fazla farkli deger varsa yalnizca sayi yazilir
DEGER_ATLA = {                   # listelemenin anlamsiz oldugu sutunlar
    "tc", "hasta_tc", "doktor_tc", "ad_soyad", "eposta", "oda_yatak",
    "giris_tarihi", "cikis_tarihi", "tarih", "goreve_baslama", "poliklinik_gunleri",
    "bashekim",
}


# Sema uretilirken doldurulur; asagidaki sonuc denetimi bunlari kullanir.
#   DEGER_INDEKSI : ASCII'ye katlanmis deger -> veritabanindaki GERCEK yazim
#                   ("uroloji" -> "Uroloji").  Model Turkce harfi dusurdugunde
#                   dogru yazimi hatirlatmak icin.
#   GRUPLANAMAZ   : her satirda benzersiz oldugu icin GROUP BY'i anlamsiz olan
#                   sutunlar (or. tani).
DEGER_INDEKSI = {}
GRUPLANAMAZ = set()

_ASCII_TR = str.maketrans("çğıöşüÇĞİÖŞÜ", "cgiosuCGIOSU")


def _katla(s):
    """Turkce harfleri ASCII karsiligina indirger + kucuk harfe cevirir."""
    return s.translate(_ASCII_TR).casefold()


def _sutun_aciklamasi(tablo, sutun):
    """Once tabloya ozel anlam, yoksa genel anlam. Bulamazsa None."""
    return SUTUN_ACIKLAMA.get(f"{tablo}.{sutun}") or SUTUN_ACIKLAMA.get(sutun)


def _deger_ozeti(cur, tablo, sutun):
    """TEXT sutunun gercek degerleri ya da kardinalite uyarisi. Yoksa None."""
    if sutun in DEGER_ATLA:
        return None
    n = cur.execute(f"SELECT COUNT(*) FROM {tablo}").fetchone()[0]
    d = cur.execute(f"SELECT COUNT(DISTINCT {sutun}) FROM {tablo}").fetchone()[0]
    if d == 0:
        return None
    # SIRA ONEMLI: once "az sayida mi" diye bak. sube_kodu gibi birincil anahtarlar
    # da d == n saglar ama degerleri (SB01..SB10) filtre yazmak icin GEREKLIDIR.
    # Benzersizlik uyarisi yalnizca listelenemeyecek kadar cok degere kalir (or. tani).
    if d <= DEGER_LISTE_SINIRI:
        vals = [r[0] for r in cur.execute(
            f"SELECT DISTINCT {sutun} FROM {tablo} WHERE {sutun} IS NOT NULL ORDER BY 1")]
        for v in vals:                        # yanlis yazimi yakalamak icin indeks
            DEGER_INDEKSI.setdefault(_katla(v), v)
        return " | ".join(vals)
    if d == n:                                # her satirda farkli -> gruplanamaz
        GRUPLANAMAZ.add(sutun)
        return f"[{d} farkli / {n} satir -> HER SATIRDA BENZERSIZ, GROUP BY ANLAMSIZ]"
    return f"[{d} farkli deger - listelenmedi, DISTINCT ile bak]"


# ARAC 1: Sema -> goreve gomulup okunur.
@tool
def get_db_schema() -> str:
    """Veritabanindaki view'leri, tablolari, sutunlari, sutun anlamlarini,
    foreign key iliskilerini ve metin sutunlarinin gercek degerlerini doner."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        cur = conn.cursor()
        cur.execute("""SELECT name, type FROM sqlite_master
                       WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%'""")
        nesneler = cur.fetchall()
        views  = [n for n, t in nesneler if t == "view"]
        tablos = [n for n, t in nesneler if t == "table"]

        parcalar = []
        degerler = {}                              # sutun adi -> deger ozeti (tek kez)
        for ad in views + tablos:                  # VIEW'ler once
            parcalar.append(("VIEW " if ad in views else "TABLO") + f" {ad}:")
            cur.execute(f"PRAGMA table_info({ad});")
            for c in cur.fetchall():
                sutun, tip = c[1], (c[2] or "")
                if sutun in GIZLI_SUTUNLAR:
                    continue
                satir = f"  {sutun} ({tip})"
                anlam = _sutun_aciklamasi(ad, sutun)
                if anlam:                          # hizali dursun -> okunurluk
                    satir = f"{satir:<34} -- {anlam}"
                parcalar.append(satir)
                # degerleri yalnizca HAM TABLODAN topla, view'de tekrarlama
                if ad in tablos and tip.upper().startswith("TEXT") and sutun not in degerler:
                    ozet = _deger_ozeti(cur, ad, sutun)
                    if ozet:
                        degerler[sutun] = ozet
            if ad in tablos:                       # FK'ler -> JOIN icin sart
                cur.execute(f"PRAGMA foreign_key_list({ad});")
                for r in cur.fetchall():
                    parcalar.append(f"  FK {ad}.{r[3]} -> {r[2]}.{r[4]}")

        parcalar.append("")
        parcalar.append("SUTUN DEGERLERI (filtre yazarken BURADAN AYNEN kopyala,")
        parcalar.append("                 Turkce harfleri degistirme):")
        for sutun, ozet in degerler.items():
            parcalar.append(f"  {sutun}: {ozet}")
    return "\n".join(parcalar)

# ─────────────────────────────────────────────────────────────────────────────
_LITERAL = re.compile(r"'([^']*)'")
_GROUPBY = re.compile(r"GROUP\s+BY\s+([A-Za-z_][A-Za-z0-9_.]*)", re.IGNORECASE)


def _dogru_yazim(lit):
    """Filtredeki deger yanlis mi yazilmis? Dogru yazimi doner, yoksa None."""
    aday = _katla(lit.strip("%").strip())
    if not aday:
        return None
    dogru = DEGER_INDEKSI.get(aday)
    if dogru:
        return dogru
    # 'Dermatoloji' -> 'Dermatoloji (Cilt Hastaliklari)' gibi kismi yazimlar;
    # yalnizca TEK aday varsa oner, yoksa belirsizdir.
    esler = {v for k, v in DEGER_INDEKSI.items() if k.startswith(aday)}
    return esler.pop() if len(esler) == 1 else None


def _sonuc_denetimi(query, sonuc):
    """Sorgu calisti ama sonuc anlamsiz/bos ise ajani uyarir (ValueError)."""
    m = _GROUPBY.search(query)
    if m:
        sutun = m.group(1).split(".")[-1]
        if sutun in GRUPLANAMAZ:
            raise ValueError(
                f"'{sutun}' sutununda her satir benzersizdir; GROUP BY {sutun} "
                f"anlamsiz (tum adetler 1 cikar, siralama rastgele olur). "
                f"'En sik / en cok {sutun}' bu veriyle HESAPLANAMAZ. "
                f"final_answer ile bunu ACIKCA soyle. Baska bir sutunla "
                f"(or. servis) gruplamayi secersen, cevabin icinde HANGI sutunu "
                f"kullandigini yazmak ZORUNDASIN; ciplak bir isim dondurme.")
    if sonuc:
        return sonuc
    for lit in _LITERAL.findall(query):        # 0 satir -> yazim hatasi mi?
        dogru = _dogru_yazim(lit)
        if dogru and dogru != lit:
            raise ValueError(
                f"Sorgu 0 satir dondurdu. Filtrede '{lit}' yazdin; "
                f"veritabanindaki gercek deger '{dogru}'. Turkce harfleri "
                f"degistirmeden AYNEN kullan ve sorguyu tekrar calistir.")
    return sonuc


# ARAC 2: SQL calistirici -> METIN DEGIL, GERCEK VERI (sozluk listesi) doner.
# Boylece ajan result[0]['sutun'] ile degeri alip final_answer'a verebilir ve DURUR.
# NOT: Sonuc SINIRLANMAZ — sorgu ne dondururse tamami gelir.
@tool
def run_sql_query(query: str) -> list:
    """Bir SELECT sorgusu calistirir ve sonuclari sozluk listesi olarak doner.

    Donus ornegi: [{'deger': 226}]  veya  [{'tc': '748...', 'ad_soyad': 'Ahmet'}, ...].
    Tek deger icin: result[0]['sutun_adi']. Sonra final_answer(deger) ile bitir.

    Args:
        query: Calistirilacak SQL (yalnizca okuma, SELECT).
    """
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        conn.row_factory = sqlite3.Row          # sonuclara sutun adiyla erisim -> dict(row)
        cur = conn.cursor()
        try:
            cur.execute(query)
        except sqlite3.OperationalError as e:
            # or. "no such table" / "attempt to write a readonly database"
            raise ValueError(f"SQL hatasi: {e}. get_db_schema ile tablo/sutun adlarini dogrula.")
        sonuc = [dict(r) for r in cur.fetchall()]  # or. [{'deger': 226}]
    return _sonuc_denetimi(query, sonuc)           # bos/anlamsiz ise ValueError

In [8]:
# 1. Modeli 

model = LiteLLMModel(
    model_id="ollama/qwen2.5-coder:7b",
    api_base="http://localhost:11434",
    temperature=0.0,
    num_ctx=12288,
)


# 2. Araclari ve Modeli Ajanta Bagliyoruz
# NOT: max_steps EKLENMEDI -> varsayilan 20 kalir; 
agent = CodeAgent(
    tools=[get_db_schema, run_sql_query],
    model=model,
    add_base_tools=False,
    verbosity_level=0,
)

In [9]:
# ══════════════════════════════════════════════════════════════════
#  Sabit kurallar — bir kez tanimlanir, her soruda kullanilir
#
#  TASARIM: sutun-duzeyi bilgi (bir sutun NE OLCER, hangi DEGERLERI alir)
#  artik burada DEGIL, semanin icinde duruyor (bkz. SUTUN_ACIKLAMA ve
#  SUTUN DEGERLERI blogu). Burada yalnizca TABLOLAR ARASI / CAPRAZ kurallar
#  kaliyor. Boylece ayni bilgi iki yerde tekrarlanmiyor.
#
#  ⚠ Bu metindeki SQL orneklerinde Turkce harfler AYNEN kullanilmali
#    ('Üroloji', 'Şifa'). ASCII'ye sadelestirilirse ornegin kendisi 0 satir
#    dondurur ve model yanlisi taklit eder. (Bu hata bir kez yasandi.)
# ══════════════════════════════════════════════════════════════════
SISTEM_KURALLARI = """Sen bir SQLite veri analistisin. Semayi kullanarak soruyu yanitla.

## 0. SUTUN SECIMI — EN ONEMLI KURAL
Semada her sutunun yaninda "--" ile NE OLCTUGU yazili.
Sutunu ADINDAN degil, BU ACIKLAMADAN sec.
Soru bir NITELIK soruyorsa ("tecrubeli", "kidemli", "verimli", "basarili",
"yogun", "yillik izni"), once o niteligi olcen sutunu aciklamalardan BUL,
sonra sorguyu yaz.
- Niteligi olcen sutun semada yoksa benzerini ZORLAMA. Uydurmak yerine sor:
      final_answer("Bu olcu veride yok. Sunlardan hangisi: gorev_yil / islem_sayisi?")
- Bir sutunu baska bir olcuye CEVIRME. Haftalik degeri 52 ile carpip yillik
  uretmek, adet sayarak yil tahmin etmek YASAK. Sutunu oldugu gibi kullan.

## 0.1 TAKMA AD (AS) KURALI
Takma ad ALDIGIN SUTUNU tarif eder, SORUNUN kelimesini degil.
    SELECT gorev_yil AS gorev_yil            <- dogru
    SELECT islem_sayisi AS tecrube_yili      <- YASAK: sutun tecrube olcmuyor
    SELECT haftalik_nobet*52 AS yillik_izin  <- YASAK: nobet izin degil
Takma adi Turkce ve sutun adina yakin tut. Emin degilsen takma ad KOYMA.
Tek deger donduren sorguda:  SELECT COUNT(*) AS deger FROM doktorlar

## 1. METIN FILTRESI — DEGERI SEMADAN KOPYALA
Semanin sonundaki "SUTUN DEGERLERI" blogunda metin sutunlarinin GERCEK
degerleri yazili. Filtre yazarken degeri oradan AYNEN kopyala.
Turkce harfleri ASCII'ye cevirme: SQLite'in buyuk/kucuk harf esitligi
yalnizca ASCII icindir, 'uroloji' ile 'Üroloji' esitlenmez.
    Kullanici "uroloji uzmani" dedi  -> listede 'Üroloji' var
        WHERE uzmanlik = 'Üroloji'          dogru
        WHERE uzmanlik LIKE '%uroloji%'     0 satir doner
    Kullanici "dermatoloji" dedi     -> listede 'Dermatoloji (Cilt Hastalıkları)'
        WHERE uzmanlik LIKE 'Dermatoloji%'  dogru
Deger listede YOKSA uydurma; en yakin degeri sor ya da bos sonucu bildir.
ad_soyad listelenmez, unvan icerir -> WHERE ad_soyad LIKE '%Ergene%'
Doktor arayacaksan uzmanlik, yatis/hasta sayacaksan servis kullan.

## 2. GRUPLANAMAYAN SUTUNLAR
Deger listesinde "HER SATIRDA BENZERSIZ" yazan sutunda (or. tani) GROUP BY
ya da "en sik / en cok" sorgusu ANLAMSIZDIR: her deger 1 kez geciyor,
donen satir rastgele olur. Boyle bir soru gelirse sayi uydurma, durumu soyle:
      final_answer("tani alaninda her kayit benzersiz (40 yatis = 40 farkli "
                   "tani); 'en sik tani' hesaplanamaz. Servis bazinda dagilim "
                   "isterseniz cikarabilirim.")

## 3. SAYMA KURALLARI
- yatislar BIR ZIYARETTIR, kisi degil. Ayni hasta birden fazla kez yatabilir.
      kac KISI  -> COUNT(DISTINCT hasta_tc)
      kac YATIS -> COUNT(*)
- hizmetler (yatisa bagli) ile doktor_islemleri (doktora bagli) AYRI
  akislardir, aralarinda bag YOKTUR. Isim benzerligine aldanma:
      "hastaya ne yapildi" -> hizmetler
      "doktor ne yapti"    -> doktor_islemleri

## 4. IKI FARKLI OLCEK — BIRLESTIRME
subeler tablosu GERCEK hastane buyuklugunu tasir (SB03 doktor_sayisi = 128).
doktorlar / hastalar / yatislar ise 40 satirlik ORNEKLEMDIR.
    "SB03'te kac doktor KAYITLI?" -> COUNT(*) FROM doktorlar WHERE ...   (5)
    "SB03'un doktor KADROSU kac?" -> SELECT doktor_sayisi FROM subeler   (128)
Soru hangisini istedigi belli degilse IKISINI DE dondur ve etiketle.
Bu iki olcegi asla toplama, bolme veya oranlama.
SUM(yatislar.fatura) ile subeler.yillik_ciro farkli olceklerdir.

## 5. SQL KURALLARI
- Sadece SELECT yaz. INSERT/UPDATE/DELETE yasak.
- SONUCLARI SINIRLAMA. Kendiliginden LIMIT EKLEME; sorunun kapsadigi TUM
  satirlari dondur. LIMIT'i yalnizca soru acikca sayi belirtiyorsa kullan
  (or. "en cok 3 doktor", "ilk 5 islem").
- Siralama isteyen sorularda OLCU SUTUNUNU da SELECT'e koy ki cevap
  dogrulanabilsin:
      SELECT ad_soyad, gorev_yil FROM v_doktor_ozet ORDER BY gorev_yil DESC
- Sutun adlarini semadan AYNEN al. Semada olmayan sutun UYDURMA.

## 6. ONCE VIEW'LERE BAK — ama ICINDE NE VAR bilerek
v_yatis_detay -> yatis + hasta + doktor + sube TEK satirda (JOIN gerekmez)
    ICINDE YOK: memnuniyet, izin_*, unvan, haftalik_nobet, gorev_yil
v_doktor_ozet -> doktor + sube + islem_sayisi / yatis_sayisi / toplam_ciro
    ICINDE VAR: gorev_yil, unvan, memnuniyet
    ICINDE YOK: izin_*, haftalik_nobet, eposta, poliklinik_gunleri
Aradigin sutun view'de yoksa ZORLAMA; ham tabloya in ve FK satirini kullan.

## 7. NASIL CALISACAKSIN
run_sql_query(sql) bir SOZLUK LISTESI doner, ornegin [{'deger': 40}].
Akis her zaman ayni:
    sonuc = run_sql_query("SELECT COUNT(*) AS deger FROM doktorlar")
    final_answer(sonuc[0]['deger'])
final_answer cagirmadan gorev BITMEZ. Sadece print etmek yeterli degildir.
SQL METNINI CEVAP OLARAK DONDURME; sorguyu run_sql_query ile CALISTIR.
Sorgu BOS liste dondurduyse "yok" deme; once filtre degerini SUTUN DEGERLERI
listesiyle karsilastir, yaziminda hata varsa duzeltip tekrar dene.

## ORNEKLER
Soru: Tecrubeli doktorlari listele.
    # tecrube = gorev_yil (semadaki aciklama). islem_sayisi DEGIL.
    sonuc = run_sql_query("SELECT ad_soyad, unvan, gorev_yil FROM v_doktor_ozet "
                          "ORDER BY gorev_yil DESC")
    final_answer(sonuc)

Soru: Yillik izni en fazla olan 5 doktor.
    # izin = izin_kalan. haftalik_nobet DEGIL, 52 ile carpma.
    sonuc = run_sql_query("SELECT ad_soyad, izin_kalan FROM doktorlar "
                          "ORDER BY izin_kalan DESC LIMIT 5")
    final_answer(sonuc)

Soru: Uroloji uzmani doktorlar kimler?
    # Deger listesinde 'Üroloji' yaziyor -> AYNEN kopyala, ASCII'ye cevirme.
    sonuc = run_sql_query("SELECT ad_soyad, unvan FROM doktorlar "
                          "WHERE uzmanlik = 'Üroloji' ORDER BY ad_soyad")
    final_answer(sonuc)

Soru: Kardiyoloji servisinde kac hasta yatti?
    sonuc = run_sql_query("SELECT COUNT(DISTINCT hasta_tc) AS deger "
                          "FROM v_yatis_detay WHERE servis = 'Kardiyoloji'")
    final_answer(sonuc[0]['deger'])

Soru: SB03 subesinde kac doktor var?
    # Belirsiz: kayitli ornek mi, gercek kadro mu -> IKISI DE dondurulur.
    kayitli = run_sql_query("SELECT COUNT(*) AS deger FROM doktorlar "
                            "WHERE sube_kodu = 'SB03'")
    kadro   = run_sql_query("SELECT doktor_sayisi AS deger FROM subeler "
                            "WHERE sube_kodu = 'SB03'")
    final_answer({"kayitli_doktor": kayitli[0]['deger'],
                  "gercek_kadro":   kadro[0]['deger']})

Soru: En cok yapilan 5 islem nedir?
    sonuc = run_sql_query("SELECT islem, COUNT(*) AS adet FROM doktor_islemleri "
                          "GROUP BY islem ORDER BY adet DESC LIMIT 5")
    final_answer(sonuc)
"""


# ══════════════════════════════════════════════════════════════════
DB_SEMA = get_db_schema()          # VIEW'ler + anlamlar + tablolar + FK'ler + degerler
print(f"Sema alindi: {len(DB_SEMA.splitlines())} satir, {len(DB_SEMA)} karakter")
print(f"Gorev metni: ~{(len(SISTEM_KURALLARI) + len(DB_SEMA)) // 3} token "
      f"(+ smolagents sistem prompt'u ~2000). num_ctx bunu karsilamali.")


def sql_cevapla(soru):
    """SQL kolu. Gorev metninin SABIT kismi (kurallar + sema) her cagride ayni;
    degisen tek sey SORU. Dagitici, etiket SQL oldugunda burayi cagirir."""
    gorev = f"""{SISTEM_KURALLARI}

## SEMA
{DB_SEMA}

## SORU
{soru}"""
    return agent.run(gorev)

Sema alindi: 121 satir, 6690 karakter
Gorev metni: ~4402 token (+ smolagents sistem prompt'u ~2000). num_ctx bunu karsilamali.


# router  ve final

In [10]:
# ═══════════════════════════════════════════════════════════════════════════
#  ROUTER v2 — SQL / RAG  (+ DIGER yalnızca regex katmanından)
#  Akış:  kural katmanı (bedava, deterministik) -> LLM -> normalize + fallback
#  LLM yalnızca SQL veya RAG döndürür. DIGER kararı sadece _KURAL_DIGER'e ait.
# ═══════════════════════════════════════════════════════════════════════════
import re
import requests

OLLAMA_HOST       = "http://localhost:11434"
ROUTER_MODEL      = "qwen3:4b-instruct-2507-q4_K_M"  # DÜŞÜNMEYEN model şart
ROUTER_KEEP_ALIVE = "5m"     # 4 GB VRAM: coder:7b/qwen3:8b yüklenince zaten atılacak
KURAL_ONCE        = True     # False -> her soru LLM'e gider (kural katmanını ölçmek için)
VARSAYILAN        = "RAG"    # LLM anlaşılmaz cevap verirse buraya düş

ETIKETLER = ("SQL", "RAG", "DIGER")   # DIGER test matrisi için duruyor

# ───────────────────────────────────────────────────────────────────────────
# 1) PROMPT — kelime listesi YOK, karar testi VAR
# ───────────────────────────────────────────────────────────────────────────
ROUTER_SYSTEM = """Sen bir sorgu yönlendiricisisin. Kullanıcının sorusunu iki sistemden
birine yönlendirirsin: SQL veya RAG.
Cevabın SADECE bu iki kelimeden biridir. Açıklama, gerekçe, noktalama yazma.
Soruyu cevaplamaya çalışma; sadece nereye gideceğine karar ver.
Her soru mutlaka SQL ya da RAG'dir; üçüncü bir seçenek yoktur."""

ROUTER_PROMPT = """## GÖREV
Soruyu iki sistemden birine yönlendir. İkisi de aynı hastane verisini görür;
aralarındaki fark verinin türü değil, cevabın KAPSAMIDIR.

SQL -> Veritabanı. Tüm kayıtları aynı anda görebilir.
       Sayma, ortalama, toplam, sıralama, kıyaslama, gruplama, süzme.
RAG -> Belge arşivi. Her seferinde yalnızca birkaç kaydı getirebilir.
       İçinde iki tür metin bulunur:
         a) Kayıt kartları — tek bir kişinin ya da birimin bilgilerinin
            bir arada durduğu künyeler.
         b) Belge metinleri — prosedür, kural, politika, yönetmelik, tanım.

## KURAL 0 — SORUNUN KONUSUNU SORGULAMA
Sorulan bilginin sende olup olmadığına, o alanın kayıtlarda bulunup
bulunmadığına, kişinin tanınıp tanınmadığına, konunun hastaneyle ilgisi
olup olmadığına BAKMA. Yalnızca kapsam kuralını uygula.

## TEMEL TEST — CEVAP İÇİN KAÇ KAYDA BAKMAK GEREKİR?
Önce şunu sor: tek bir kaydın içine bakmak yeter mi, yoksa çok sayıda
kaydın taranması mı gerekir?

Çok sayıda kaydın taranması gerekiyorsa -> SQL
  Sayma, ortalama, toplam, gruplama, süzme; ayrıca kıyaslama, sıralama,
  ortalamayla karşılaştırma, bir sıradaki yer: "kaçıncı", "en ...",
  "ortalamanın üstünde mi", "-den fazla mı". Soruda tek bir isim
  geçmesi bunu değiştirmez.
Adı, kodu ya da kimliği verilmiş BİR (en fazla iki) kişi/birim
hakkında, o kişiye/birime ait bir bilgi soruluyorsa                -> RAG
Cevap bir tanım, kural, prosedür, politika ya da normsa             -> RAG
Kararsız kaldıysan                                                  -> RAG

## KURAL 1 — SAYI VEYA "KAÇ" TEK BAŞINA SQL DEMEK DEĞİLDİR
Cevabın sayı olması yönlendirmeyi belirlemez; kapsam belirler.
Tek bir kişiye ya da birime ait sayı RAG'dir.

## KURAL 2 — FİİL KİPİ
Olması gerekeni soran kip her zaman RAG'dir; cevap bir kural metnindedir.
("olmalıdır, tutmalı, yazılabilir, düşmelidir")
Gerçekleşmiş bir durumu soran kip için ("tutuyor, yaptı, kaldı, kullandı")
kapsam kuralını uygula.

## KURAL 3 — KIYAS VE SIRALAMA HER ZAMAN SQL'DİR
Soru tek bir kişiyi anıyor olsa bile, onu başkalarıyla kıyaslamayı,
sıralamayı veya bir sıradaki yerini bulmayı gerektiriyorsa SQL'dir.

## KURAL 4 — KARMA SORULAR
Soru hem tek kayıt hem küme bilgisi istiyorsa, cevabın AĞIRLIĞI
hangisindeyse onu seç.

## ÖRNEKLER — sınıra yakın olanlar seçilmiştir

# Sayı sorusu, ama tek kişiye ait
Soru: Dr. Ergene'nin kaç gün izni kaldı?                      Cevap: RAG
Soru: Doktorların ortalama kalan izni kaç gün?                Cevap: SQL

# Tek kişi anılıyor, ama kıyas isteniyor
Soru: Dr. Batur'un memnuniyet puanı kaç?                      Cevap: RAG
Soru: Dr. Batur en yüksek memnuniyete sahip doktor mu?        Cevap: SQL
Soru: Dr. Alkan'ın memnuniyeti ortalamanın üstünde mi?        Cevap: SQL
Soru: Dr. Sarıkaya kaçıncı en kıdemli doktor?                 Cevap: SQL

# Grup adı geçiyor — tek kayıt değil, küme
Soru: Dr. Alkan'ın uzmanlığı ne?                              Cevap: RAG
Soru: Kardiyolojideki doktorlar kimler?                       Cevap: SQL

# İçinde sayı var ama kural soruluyor
Soru: Bir hastaya en fazla kaç gün ilaç yazılabilir?          Cevap: RAG
Soru: Bu ay kaç hastaya ilaç yazıldı?                         Cevap: SQL

# İçinde toplama kelimesi var ama gerekçe soruluyor
Soru: Ortalama yatış süresi neden takip edilir?               Cevap: RAG
Soru: Ortalama yatış süresi kaç gün?                          Cevap: SQL

# "kimler/kimlere" — yetki kuralı mı, kişi listesi mi?
Soru: Reçete yazma yetkisi kimlere aittir?                    Cevap: RAG
Soru: Bu ay en çok reçete yazan doktorlar kimler?             Cevap: SQL

# "nasıl" — anlatım mı, kayıt mı, küme mi?
Soru: Taburcu süreci nasıl yürütülür?                         Cevap: RAG
Soru: Dr. Sarıkaya'nın performansı nasıl?                     Cevap: RAG
Soru: Doktorların genel performansı nasıl?                    Cevap: SQL

# Kapsam, kaydın türünden ve konudan bağımsızdır
Soru: Dr. Çalışkan'ın son randevusu ne zamandı?               Cevap: RAG
Soru: Bu ay kaç randevu oluşturuldu?                          Cevap: SQL
Soru: Mehmet Öz'ün maaşı ne kadar?                            Cevap: RAG

Soru: {soru}
Cevap:"""

# ───────────────────────────────────────────────────────────────────────────
# 2) KURAL KATMANI — sadece yüksek kesinlikli kalıplar, gerisi LLM'e

_KURAL_DIGER = re.compile(
    r"^\s*(merhaba|selam(lar)?|g[üu]nayd[ıi]n|iyi (g[üu]nler|ak[şs]amlar)|"
    r"nas[ıi]ls[ıi]n|naber|te[şs]ekk[üu]r|sa[ğg]ol|eyvallah|"
    r"g[öo]r[üu][şs][üu]r[üu]z|ho[şs][çc]a kal)\b"
    r"|\b(sen kimsin|kimsin sen|ne(ler)? yapabilir(sin|iyorsun)|ad[ıi]n ne|"
    r"nas[ıi]l [çc]al[ıi][şs][ıi]yorsun)\b",
    re.IGNORECASE,
)

# Norm / kural kipi + belge alanı isimleri -> kesin RAG (aggregation kelimesi olsa bile)
_KURAL_RAG = re.compile(
    r"\b\w*(olmal[ıi](d[ıi]r)?|d[üu][şs]meli(dir)?|gerekir|gerekmektedir|"
    r"zorunludur|yasakt[ıi]r|yaz[ıi]labilir|yap[ıi]labilir|verilmeli(dir)?|"
    r"uygulanmal[ıi](d[ıi]r)?|edilmeli(dir)?)\b"
    r"|\b(prosed[üu]r|politika|y[öo]netmelik|mevzuat|talimat|protokol|"
    r"kural(lar[ıi])?|standart)\b",
    re.IGNORECASE,
)

# Toplama fiili + mevcut kayıt kipi -> kesin SQL
_KURAL_SQL = re.compile(
    r"\b(ka[çc]|toplam|ortalama|say[ıi]s[ıi]|listele|s[ıi]rala|"
    r"en (?:[çc]ok|az|fazla|y[üu]ksek|d[üu][şs][üu]k|ya[şs]l[ıi]|gen[çc]))\b"
    r"[\s\S]{0,60}?"
    r"\b(var(d[ıi]r)?|kay[ıi]tl[ıi]|[çc]al[ıi][şs][ıi]yor|oldu|yatt[ıi]|"
    r"bulunuyor|mevcut|ald[ıi]|yapt[ıi])\b",
    re.IGNORECASE,
)


def _kural_karari(soru):
    """Etiket döner ya da None (= LLM'e sor). Sıra önemli: DIGER -> RAG -> SQL."""
    if _KURAL_DIGER.search(soru):
        return "DIGER"
    if _KURAL_RAG.search(soru):      # norm kipi, aggregation kelimesini EZER
        return "RAG"
    if _KURAL_SQL.search(soru):
        return "SQL"
    return None


# ───────────────────────────────────────────────────────────────────────────
# 3) ÇIKTI NORMALİZASYONU — LLM'den yalnızca SQL / RAG kabul edilir
# ───────────────────────────────────────────────────────────────────────────
_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_ETIKET = re.compile(r"\b(SQL|RAG)\b", re.IGNORECASE)   # DIGER bilerek yok


def _normalize(ham, soru):
    m = _ETIKET.search(_THINK.sub("", ham or ""))
    if m:
        return m.group(1).upper()
    # LLM saçmaladı ya da DIGER dedi -> kural katmanına, oradan varsayılana düş
    return _kural_karari(soru) or VARSAYILAN


# ───────────────────────────────────────────────────────────────────────────
# 4) ANA FONKSİYON
# ───────────────────────────────────────────────────────────────────────────
_CACHE = {}


def route(soru, ayrinti=False):
    """Soruyu SQL / RAG / DIGER etiketlerinden birine yönlendirir.
    DIGER yalnızca kural katmanından gelebilir; LLM SQL ya da RAG der.

    ayrinti=False -> "SQL"
    ayrinti=True  -> {"etiket":..., "kaynak":..., "sure":..., "token":...}

    'sure' Ollama'nın total_duration alanından gelir (ns -> s). Kural/cache
    yolunda hiç istek gitmediği için 0.0'dır.
    """
    anahtar = " ".join(soru.lower().split())
    if anahtar in _CACHE:
        sonuc = dict(_CACHE[anahtar], kaynak="cache", sure=0.0)
        return sonuc if ayrinti else sonuc["etiket"]

    if KURAL_ONCE:
        etiket = _kural_karari(soru)
        if etiket:
            sonuc = {"etiket": etiket, "kaynak": "kural", "sure": 0.0, "token": 0}
            _CACHE[anahtar] = {"etiket": etiket, "token": 0}
            return sonuc if ayrinti else etiket

    r = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={
            "model": ROUTER_MODEL,
            "system": ROUTER_SYSTEM,
            "prompt": ROUTER_PROMPT.format(soru=soru),
            "stream": False,
            "think": False,
            "keep_alive": ROUTER_KEEP_ALIVE,
            "options": {
                "temperature": 0.0,
                "top_k": 1,          
                "top_p": 1.0,
                "seed": 42,
                "num_ctx": 2048,
                "stop": ["\n", "Soru:", "Cevap:"],
            },
        },
        timeout=120,
    )
    r.raise_for_status()
    d = r.json()

    sure = d.get("total_duration", 0) / 1e9        # ns -> s (model yükleme dahil)
    etiket = _normalize(d.get("response", ""), soru)
    token = d.get("prompt_eval_count", 0) + d.get("eval_count", 0)

    _CACHE[anahtar] = {"etiket": etiket, "token": token}
    sonuc = {"etiket": etiket, "kaynak": "llm", "sure": sure, "token": token}
    return sonuc if ayrinti else etiket



# ── tek sorgu ──────────────────────────────────────────────────────────────
SORGU = "kardiyoloji ameliyatlarını listele"

d = route(SORGU, ayrinti=True)
print(f"{d['etiket']} | {d['kaynak']} | {d['sure']}s | {d['token']}")

SQL | llm | 8.338272s | 1567


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
#  DAGITICI — sistemin tek giris noktasi
#  route() yalnizca ETIKET uretir; isi yapan yer burasidir:
#      SQL   -> sql_cevapla(soru)      -> CodeAgent + sqlite   (sql query hucresi)
#      RAG   -> answer_question(soru)  -> chroma + qwen3:8b    (rag query hucresi)
#      DIGER -> hicbir kol calismaz, ekrana sadece "DIGER" yazilir
#
#  Donus: (etiket, cevap).  DIGER'de cevap None'dir.
#  ONKOSUL: yukaridaki TUM hucreler calistirilmis olmali (RAG fonksiyonlari,
#  ajan, router). Python global'leri cagri aninda cozer; eksik olan varsa
#  hata cagirinca gelir: NameError.
# ═══════════════════════════════════════════════════════════════════════════
def sor(soru):
    d = route(soru, ayrinti=True)          # ayrinti=True -> kaynak/sure de gelsin
    etiket = d["etiket"]
    print(f"[ROUTER] {etiket}  ({d['kaynak']}, {d['sure']:.2f}s)")

    if etiket == "SQL":
        return etiket, sql_cevapla(soru)

    if etiket == "RAG":
        cevap, flag = answer_question(soru)
        if flag:                           # BILGI_BULUNAMADI: RAG bulamadi
            print(f"[FLAG: {flag}]")
        return etiket, cevap

    print("DIGER")                         # su an bu kolda henüz bir şey eklemedim
    return etiket, None


# ── kullanim ──────────────────────────────────────────────────────────────
etiket, cevap = sor("yıllık izni en fazla olan 5 doktoru listele?")
if cevap is not None:
    print("\n--- SONUC ---")
    print(cevap)

[ROUTER] SQL  (llm, 0.50s)

--- SONUC ---
[{'ad_soyad': 'Prof. Dr. Leman Özkul', 'izin_kalan': 20}, {'ad_soyad': 'Prof. Dr. Ebru Yaman', 'izin_kalan': 17}, {'ad_soyad': 'Op. Dr. Arda Tunçel', 'izin_kalan': 16}, {'ad_soyad': 'Doç. Dr. Bahadır Küçükoğlu', 'izin_kalan': 15}, {'ad_soyad': 'Uzm. Dr. Bengi Yalvaç', 'izin_kalan': 15}]


In [12]:
# ── AYARLAR ───────────────────────────────────────────────────────────────
HOST = "127.0.0.1"
PORT = 8000
# ──────────────────────────────────────────────────────────────────────────
import threading
import uvicorn
import time
import traceback
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from query_logger import log_sorgu

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class Soru(BaseModel):
    soru: str

def _format_cevap(cevap_raw):
    if cevap_raw is None:
        return None
    if isinstance(cevap_raw, str):
        return cevap_raw
    if isinstance(cevap_raw, (int, float, bool)):
        return str(cevap_raw)
    if isinstance(cevap_raw, list):
        if len(cevap_raw) == 0:
            return "Sorgu sonucunda herhangi bir kayıt bulunamadı."
        if all(isinstance(item, dict) for item in cevap_raw):
            lines = []
            for i, row in enumerate(cevap_raw, 1):
                row_str = " | ".join(f"{k}: {v}" for k, v in row.items())
                lines.append(f"{i}. {row_str}")
            return "\n".join(lines)
        return "\n".join(f"• {item}" for item in cevap_raw)
    if isinstance(cevap_raw, dict):
        return "\n".join(f"{k}: {v}" for k, v in cevap_raw.items())
    return str(cevap_raw)

@app.post("/sor")
def sor_api(istek: Soru):
    t0 = time.time()
    etiket = None
    hata_asamasi = None
    try:
        hata_asamasi = "ROUTER"
        d = route(istek.soru, ayrinti=True)
        etiket = d.get("etiket")
        
        if etiket == "SQL":
            hata_asamasi = "SQL_AGENT"
            raw_cevap = sql_cevapla(istek.soru)
            cevap = _format_cevap(raw_cevap)
        elif etiket == "RAG":
            hata_asamasi = "RAG_SEARCH"
            raw_cevap, flag = answer_question(istek.soru)
            cevap = _format_cevap(raw_cevap)
        else:
            cevap = "Bu soru için herhangi bir yanıt eşleşmedi."
            
        sure = time.time() - t0
        log_sorgu(soru=istek.soru, etiket=etiket, cevap=str(cevap), sure_sn=sure)
        return {"etiket": etiket, "cevap": cevap, "hata": None}
    except Exception as e:
        sure = time.time() - t0
        hata_msg = f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
        log_sorgu(soru=istek.soru, etiket=etiket, hata_asamasi=hata_asamasi, hata_detayi=hata_msg, sure_sn=sure)
        return {"etiket": etiket, "cevap": None, "hata": str(e)}

threading.Thread(
    target=lambda: uvicorn.run(app, host=HOST, port=PORT, log_level="warning"),
    daemon=True,
).start()
print(f"API ve Loglama Aktif: http://{HOST}:{PORT}/sor")

# python serve_web.py



API ve Loglama Aktif: http://127.0.0.1:8000/sor


[1/3] Sorgu embedding'e çevriliyor ve Chroma'da en yakın 5 parça aranıyor...
      -> 5 parça bulundu.
[2/3] Bağlam ve prompt oluşturuluyor...
[3/3] Ollama'ya gönderiliyor (model: qwen3:8b)... Bu adım uzun sürebilir.
      -> Cevap alındı.

